# CRSP daily market-data audit

This notebook audits the fixed CRSP daily-stock and market-index snapshots downloaded by `scripts/11_download_crsp_market_data.py`.

It checks:

1. the structure and temporal coverage of both extracts;
2. key integrity, numeric domains, security coverage, and calendar agreement;
3. missing stock observations and their CRSP data-quality flags; and
4. whether the fixed date range supports the historical feature and forward-outcome windows.

Manifest and SHA-256 verification is performed during download and is not repeated here. The notebook reads the saved files only.

## 1. Setup and inputs

In [1]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.parquet as pq


project_root = Path.cwd().resolve()

while not (project_root / "config" / "paths.yaml").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not locate the project root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.configuration import get_crsp_config, get_paths_config


paths = get_paths_config()
source = get_crsp_config()
daily_path = paths.raw / "crsp" / "daily_stock.parquet"
index_path = paths.raw / "crsp" / "daily_market_index.parquet"
security_nodes_path = (
    paths.interim / "networks" / "ownership" / "security_nodes.parquet"
)

table_paths = {
    "daily_stock": daily_path,
    "daily_market_index": index_path,
    "security_nodes": security_nodes_path,
}

for name, path in table_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"Required {name} file not found: {path}")

inventory = pd.DataFrame(
    {
        "table": table_paths.keys(),
        "rows": [
            pq.ParquetFile(path).metadata.num_rows
            for path in table_paths.values()
        ],
    }
)

connection = duckdb.connect(database=":memory:")
connection.read_parquet(str(daily_path)).create_view("daily_stock")
connection.read_parquet(str(index_path)).create_view("market_index")
connection.read_parquet(str(security_nodes_path)).create_view("security_nodes")

display(inventory.style.format({"rows": "{:,}"}))

,table,rows
0,daily_stock,"14,791,343"
1,daily_market_index,"3,665"
2,security_nodes,"206,868"


## 2. Structure and coverage

In [2]:
schema = pd.concat(
    [
        pd.DataFrame(
            [(field.name, str(field.type)) for field in pq.read_schema(path)],
            columns=["column", "type"],
        ).assign(table=name)
        for name, path in {
            "daily_stock": daily_path,
            "daily_market_index": index_path,
        }.items()
    ],
    ignore_index=True,
)[['table', 'column', 'type']]

coverage = connection.execute(
    '''
    SELECT
        'daily_stock' AS table_name,
        COUNT(*) AS rows,
        COUNT(DISTINCT permno) AS identifiers,
        COUNT(DISTINCT dlycaldt) AS trading_dates,
        MIN(dlycaldt) AS first_date,
        MAX(dlycaldt) AS last_date
    FROM daily_stock

    UNION ALL

    SELECT
        'daily_market_index',
        COUNT(*),
        COUNT(DISTINCT indno),
        COUNT(DISTINCT dlycaldt),
        MIN(dlycaldt),
        MAX(dlycaldt)
    FROM market_index
    '''
).fetchdf()

display(schema)
display(coverage.style.format({
    "rows": "{:,}",
    "identifiers": "{:,}",
    "trading_dates": "{:,}",
}))

,table,column,type
0,daily_stock,permno,int64
1,daily_stock,permco,int64
2,daily_stock,dlycaldt,timestamp[ns]
3,daily_stock,conditionaltype,string
4,daily_stock,tradingstatusflg,string
5,daily_stock,dlydelflg,string
6,daily_stock,dlyprc,double
7,daily_stock,dlyprcflg,string
8,daily_stock,dlycap,double
9,daily_stock,dlycapflg,string


,table_name,rows,identifiers,trading_dates,first_date,last_date
0,daily_stock,"14,791,343","7,232","3,665",2012-01-03 00:00:00,2026-07-31 00:00:00
1,daily_market_index,"3,665",1,"3,665",2012-01-03 00:00:00,2026-07-31 00:00:00


## 3. Integrity checks

The stock file should contain one row per PERMNO-date, cover exactly the securities needed by the quarterly ownership networks, and share its trading calendar with the market index. Numeric checks retain missing observations but reject impossible or non-finite nonmissing values.

In [3]:
daily_start = source.daily_stock.start_date.isoformat()
daily_end = source.daily_stock.end_date.isoformat()
market_index_id = source.market_index_id

check_values = connection.execute(
    f'''
    WITH daily AS (
        SELECT
            COUNT(*) - COUNT(DISTINCT (permno, dlycaldt))
                AS duplicate_stock_keys,
            COUNT(*) FILTER (WHERE permno IS NULL OR dlycaldt IS NULL)
                AS missing_stock_keys,
            COUNT(*) FILTER (WHERE permno <= 0) AS invalid_permnos,
            COUNT(*) FILTER (
                WHERE dlycaldt < DATE '{daily_start}'
                   OR dlycaldt > DATE '{daily_end}'
            ) AS stock_dates_outside_config,
            COUNT(*) FILTER (
                WHERE dlyret IS NOT NULL
                  AND (NOT isfinite(dlyret) OR dlyret < -1)
            ) AS invalid_stock_returns,
            COUNT(*) FILTER (
                WHERE (dlyprc IS NOT NULL AND NOT isfinite(dlyprc))
                   OR (dlycap IS NOT NULL AND (NOT isfinite(dlycap) OR dlycap < 0))
                   OR (dlyvol IS NOT NULL AND (NOT isfinite(dlyvol) OR dlyvol < 0))
                   OR (shrout IS NOT NULL AND shrout < 0)
            ) AS invalid_stock_values
        FROM daily_stock
    ),
    market AS (
        SELECT
            COUNT(*) - COUNT(DISTINCT dlycaldt) AS duplicate_index_dates,
            COUNT(*) FILTER (WHERE indno != {market_index_id})
                AS unexpected_index_ids,
            COUNT(*) FILTER (
                WHERE dlycaldt IS NULL
                   OR dlytotret IS NULL
                   OR NOT isfinite(dlytotret)
                   OR dlytotret < -1
            ) AS invalid_index_rows
        FROM market_index
    ),
    required_permnos AS (
        SELECT DISTINCT permno FROM security_nodes
    ),
    daily_permnos AS (
        SELECT DISTINCT permno FROM daily_stock
    ),
    universe AS (
        SELECT
            COUNT(*) FILTER (WHERE d.permno IS NULL) AS missing_required_permnos,
            COUNT(*) FILTER (WHERE r.permno IS NULL) AS unexpected_daily_permnos
        FROM required_permnos r
        FULL JOIN daily_permnos d USING (permno)
    ),
    daily_dates AS (
        SELECT DISTINCT dlycaldt FROM daily_stock
    ),
    index_dates AS (
        SELECT DISTINCT dlycaldt FROM market_index
    ),
    calendar AS (
        SELECT COUNT(*) AS calendar_mismatches
        FROM daily_dates
        FULL JOIN index_dates USING (dlycaldt)
        WHERE daily_dates.dlycaldt IS NULL
           OR index_dates.dlycaldt IS NULL
    )
    SELECT * FROM daily CROSS JOIN market CROSS JOIN universe CROSS JOIN calendar
    '''
).fetchdf().iloc[0]

integrity_checks = pd.DataFrame(
    {
        "check": check_values.index.str.replace("_", " "),
        "violations": check_values.values,
    }
)

display(integrity_checks.style.format({"violations": "{:,}"}))
print()
print(f"Total integrity violations: {integrity_checks['violations'].sum():,}")

,check,violations
0,duplicate stock keys,0
1,missing stock keys,0
2,invalid permnos,0
3,stock dates outside config,0
4,invalid stock returns,0
5,invalid stock values,0
6,duplicate index dates,0
7,unexpected index ids,0
8,invalid index rows,0
9,missing required permnos,0



Total integrity violations: 0


## 4. Missing returns and CRSP flags

Missing returns are preserved rather than replaced with zero. CRSP's status and missing-return flags indicate why observations may not contain an ordinary return.

In [4]:
missing_counts = connection.execute(
    '''
    SELECT
        COUNT(*) AS rows,
        COUNT(*) - COUNT(dlyret) AS dlyret,
        COUNT(*) - COUNT(dlyprc) AS dlyprc,
        COUNT(*) - COUNT(dlycap) AS dlycap,
        COUNT(*) - COUNT(dlyvol) AS dlyvol,
        COUNT(*) - COUNT(shrout) AS shrout
    FROM daily_stock
    '''
).fetchdf().iloc[0]

missingness = pd.DataFrame(
    {
        "column": missing_counts.index[1:],
        "missing_rows": missing_counts.values[1:],
        "missing_share": missing_counts.values[1:] / missing_counts["rows"],
    }
)

missing_return_flags = connection.execute(
    '''
    SELECT
        COALESCE(dlyretmissflg, '<missing>') AS return_missing_flag,
        COALESCE(conditionaltype, '<missing>') AS condition,
        COALESCE(tradingstatusflg, '<missing>') AS trading_status,
        COALESCE(dlydelflg, '<missing>') AS delisting_flag,
        COUNT(*) AS rows
    FROM daily_stock
    WHERE dlyret IS NULL
    GROUP BY ALL
    ORDER BY rows DESC
    '''
).fetchdf()

display(missingness.style.format({
    "missing_rows": "{:,}",
    "missing_share": "{:.2%}",
}))
display(missing_return_flags.style.format({"rows": "{:,}"}))

,column,missing_rows,missing_share
0,dlyret,"152,018",1.03%
1,dlyprc,"148,165",1.00%
2,dlycap,"151,624",1.03%
3,dlyvol,"151,648",1.03%
4,shrout,"3,509",0.02%


,return_missing_flag,condition,trading_status,delisting_flag,rows
0,NT,NT,X,N,"144,893"
1,NS,RW,A,N,"3,599"
2,MP,RW,S,N,"1,531"
3,MP,RW,A,N,"1,470"
4,MP,RW,H,N,221
5,RA,RW,A,N,194
6,DM,N/A,D,Y,41
7,GP,RW,A,N,36
8,DG,N/A,D,Y,19
9,DP,N/A,D,Y,14


## 5. Feature and outcome windows

For each quarterly information date, historical features use trading days strictly before that date and the primary outcome uses trading days strictly after it. The fixed extract can support these windows only when it contains at least 252 prior and 63 subsequent market days; stock-specific eligibility is assessed later from each security's available returns.

In [5]:
window_coverage = connection.execute(
    '''
    WITH prediction_dates AS (
        SELECT DISTINCT report_period, information_date
        FROM security_nodes
    )
    SELECT
        p.report_period,
        p.information_date,
        COUNT(*) FILTER (WHERE i.dlycaldt < p.information_date)
            AS prior_market_days,
        COUNT(*) FILTER (WHERE i.dlycaldt > p.information_date)
            AS subsequent_market_days
    FROM prediction_dates p
    CROSS JOIN market_index i
    GROUP BY ALL
    ORDER BY p.report_period
    '''
).fetchdf()

window_summary = pd.DataFrame(
    {
        "quarterly_information_dates": [len(window_coverage)],
        "minimum_prior_market_days": [window_coverage["prior_market_days"].min()],
        "dates_with_fewer_than_252_prior_days": [
            (window_coverage["prior_market_days"] < 252).sum()
        ],
        "dates_with_fewer_than_63_subsequent_days": [
            (window_coverage["subsequent_market_days"] < 63).sum()
        ],
    }
)
incomplete_outcomes = window_coverage.loc[
    window_coverage["subsequent_market_days"] < 63
]

display(window_summary.style.format("{:,}"))
display(incomplete_outcomes.style.format({
    "prior_market_days": "{:,}",
    "subsequent_market_days": "{:,}",
}))

connection.close()

,quarterly_information_dates,minimum_prior_market_days,dates_with_fewer_than_252_prior_days,dates_with_fewer_than_63_subsequent_days
0,52,405,0,1


,report_period,information_date,prior_market_days,subsequent_market_days
51,2026-03-31 00:00:00,2026-05-15 00:00:00,"3,612",52


## 6. Interpretation

- **The two fixed extracts have the required coverage.** The stock file contains 14,791,343 rows for 7,232 PERMNOs, while the index contains one observation on each of the 3,665 trading dates from 3 January 2012 through 31 July 2026.
- **The files are internally consistent.** All integrity checks return zero: keys are unique and complete, values satisfy their basic domains, the PERMNO universe matches the quarterly security nodes, and the stock and index calendars agree exactly.
- **Missing returns must remain missing.** The stock file contains 152,018 missing returns (1.03% of rows), and every case carries CRSP condition, trading-status, or missing-return information. Most are non-trading records; replacing them with zero would incorrectly treat unavailable returns as observed no-change days.
- **Historical coverage is sufficient.** Every quarterly information date has at least 405 earlier market days, so the raw extract supports the 252-day historical window.
- **The final forward window is incomplete.** The 15 May 2026 information date has only 52 subsequent market days in the fixed extract. Consequently, the 2026 Q1 observation cannot supply the primary 63-day target and must be excluded at the target-construction stage.

The raw CRSP market data are therefore suitable for point-in-time feature and outcome construction. This audit establishes source coverage; later pipeline stages determine usability separately for each stock-quarter.